# Day 23 — Challenge 1: adaptive spectral measurements

This experiment keeps **30 measurements and equal acquisition durations**. Ten initial measurements cover the range. Subsequent measurements use a Gaussian process to maximize expected reduction of uncertainty across the 512-channel spectrum. Every fifth adaptive measurement fills the largest remaining coverage gap, guarding against overlooked peaks.

The original organizer notebook is unchanged. This notebook only handles Challenge 1. No hidden spectra or noise laws are used. Adaptive placement is a hypothesis to evaluate, not a guaranteed improvement.

Run in an environment with `numpy`, `scipy`, `scikit-learn`, `matplotlib`, and `requests` (for example Colab). The optional installation cell is commented out. API access stays disabled until you change `RUN_LIVE` below. Submission is separately disabled.


In [ ]:
# %pip install numpy scipy scikit-learn matplotlib requests

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import requests
from scipy.linalg import solve_triangular
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

N_TOTAL = 30
N_INITIAL = 10
RUN_LIVE = False
BASE_URL = "https://athena-spectral-api.onrender.com"


def api(method, path, payload=None, token=None):
    response = requests.request(
        method, BASE_URL + path, json=payload,
        headers={"X-Session-ID": token} if token else {}, timeout=120,
    )
    response.raise_for_status()
    return response.json()


## Model and placement rule

Coordinates are scaled to [0, 1]. A Matérn 3/2 kernel models the signal; a white-noise component estimates a common observation-noise variance. Equal durations do not guarantee equal noise, so this is an initial modeling assumption.

The signal length scale is constrained between two channel spacings and half the full range. These bounds are assumptions to revisit if narrow features are suspected. Acquisition uses **latent signal covariance**, excluding white measurement noise. For each candidate, it sums the expected variance reduction over the whole grid. The occasional coverage step is independent of model confidence.

With fixed kernel parameters, GP uncertainty does not depend on measured intensities. Here the kernel is refitted after each measurement, allowing observed values to inform later decisions.


In [ ]:
def fit_model(grid, observations):
    u = (grid - grid[0]) / (grid[-1] - grid[0])
    measured = np.array([r["wavelength"] for r in observations])
    X = ((measured - grid[0]) / (grid[-1] - grid[0]))[:, None]
    y = np.array([r["intensity"] for r in observations])
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(length_scale=0.08,
                 length_scale_bounds=(2 / (len(grid) - 1), 0.5), nu=1.5)
        + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-6, 1.0))
    )
    gp = GaussianProcessRegressor(
        kernel=kernel, normalize_y=True, alpha=1e-8,
        n_restarts_optimizer=3, random_state=23,
    ).fit(X, y)
    U = u[:, None]
    signal_kernel = gp.kernel_.k1
    cross = signal_kernel(U, X)
    v = solve_triangular(gp.L_, cross.T, lower=True)
    # Covariance in normalized intensity units, with observation noise excluded.
    covariance = signal_kernel(U) - v.T @ v
    covariance = (covariance + covariance.T) / 2
    variance = np.maximum(np.diag(covariance), 0)
    scale = float(np.std(y))
    if scale == 0:
        scale = 1.0
    mean = gp.predict(U)
    std = np.sqrt(variance) * scale
    return gp, mean, std, covariance


def choose_next(grid, observations):
    gp, mean, std, covariance = fit_model(grid, observations)
    measured = np.array([r["wavelength"] for r in observations])
    distances = np.min(abs(grid[:, None] - measured[None, :]), axis=1)
    available = distances > (grid[-1] - grid[0]) * 1e-10
    adaptive_number = len(observations) - N_INITIAL + 1
    if adaptive_number % 5 == 0:
        scores = distances.copy()
        rule = "coverage"
    else:
        noise = gp.kernel_.k2.noise_level + gp.alpha
        denominator = np.maximum(np.diag(covariance), 0) + noise
        scores = np.sum(covariance ** 2, axis=0) / denominator
        rule = "variance reduction"
    scores[~available] = -np.inf
    return int(np.argmax(scores)), rule


## Collect measurements — explicit live switch

Set `RUN_LIVE = True` only when ready. A new episode is created once in this kernel. Re-running the cell resumes the same in-memory observations and refuses to exceed 30. Each successful response is saved immediately to a local checkpoint; after a kernel restart, inspect that checkpoint and restore the session deliberately rather than blindly starting over.

Network failures can occur after a server has charged a measurement. Requests are **not automatically retried**. If one fails, inspect `/budget` and reconcile the session before continuing. The remaining budget is displayed at completion; its response schema is not assumed here. Do not run the original baseline in this session.


In [ ]:
if not RUN_LIVE:
    print("Live acquisition disabled. Set RUN_LIVE = True when ready.")
else:
    if "adaptive_state" not in globals():
        info = api("GET", "/info")
        energy = np.linspace(*info["wavelength_range"], info["channels"])
        assert len(energy) >= N_TOTAL and np.all(np.diff(energy) > 0)
        episode = api("POST", "/session", {"task": "single-spectrum"})
        adaptive_state = {
            "session_id": episode["session_id"],
            "time_budget": float(episode["time_budget"]),
            "energy": energy.tolist(), "observations": [],
        }
    energy = np.asarray(adaptive_state["energy"])
    observations = adaptive_state["observations"]
    token = adaptive_state["session_id"]
    duration = adaptive_state["time_budget"] / N_TOTAL
    checkpoint = Path("challenge1_adaptive_checkpoint.json")
    checkpoint.write_text(json.dumps(adaptive_state, indent=2))
    initial_indices = np.rint(np.linspace(0, len(energy) - 1, N_INITIAL)).astype(int)
    while len(observations) < N_TOTAL:
        # The original notebook exposes /budget but does not document its field names.
        # Display it for audit; server-side enforcement remains authoritative.
        if len(observations) < N_INITIAL:
            idx, rule = int(initial_indices[len(observations)]), "initial coverage"
        else:
            idx, rule = choose_next(energy, observations)
        result = api("POST", "/pixel", {
            "wavelength": float(energy[idx]), "time": float(duration),
        }, token)
        record = dict(result)
        record.update(requested_wavelength=float(energy[idx]),
                      requested_time=float(duration), placement_rule=rule)
        observations.append(record)
        checkpoint.write_text(json.dumps(adaptive_state, indent=2))
        print(f"{len(observations):2d}/30: E={energy[idx]:.4f}, {rule}")
    print("Remaining budget:", api("GET", "/budget", token=token))


## Reconstruct and inspect

The shaded band is approximate model uncertainty about the noiseless spectrum, conditional on fitted kernel parameters. It is not a verified error bound. Inspect fit warnings and kernel parameters; estimates at parameter bounds can indicate model mismatch.


In [ ]:
if "adaptive_state" in globals() and len(adaptive_state["observations"]) == N_TOTAL:
    energy = np.asarray(adaptive_state["energy"])
    observations = adaptive_state["observations"]
    gp, predicted_spectrum1, signal_std, _ = fit_model(energy, observations)
    assert predicted_spectrum1.shape == (len(energy),)
    assert np.isfinite(predicted_spectrum1).all()
    measured_E1 = np.array([r["wavelength"] for r in observations])
    measured_I1 = np.array([r["intensity"] for r in observations])
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(energy, predicted_spectrum1, label="GP prediction")
    ax.fill_between(energy, predicted_spectrum1 - 1.96 * signal_std,
                    predicted_spectrum1 + 1.96 * signal_std,
                    alpha=0.2, label="Approx. 95% latent-signal interval")
    points = ax.scatter(measured_E1, measured_I1,
                        c=np.arange(1, N_TOTAL + 1), cmap="viridis", zorder=3)
    fig.colorbar(points, ax=ax, label="Measurement order")
    ax.set(xlabel="Spectral coordinate (a.u.)", ylabel="Intensity (a.u.)",
           title="Challenge 1: 30 equal-duration adaptive measurements")
    ax.legend()
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
    print("Fitted kernel:", gp.kernel_)
    submission1 = {"energy": energy.tolist(),
                   "predicted_intensity": predicted_spectrum1.tolist()}
    Path("challenge1_adaptive_prediction.json").write_text(json.dumps(submission1))
else:
    print("Complete all 30 measurements before reconstruction.")


## Optional final submission

Submission locks the episode. Leave disabled until you have reviewed the prediction.


In [ ]:
SUBMIT = False
if SUBMIT:
    assert len(adaptive_state["observations"]) == N_TOTAL
    result1 = api("POST", "/submit", submission1, adaptive_state["session_id"])
    print(result1)


## Fair comparison with the baseline

Use separate episodes and equal total budgets. To distinguish the effect of placement from the effect of reconstruction, compare:

1. Uniform placement + linear interpolation (organizer baseline).
2. Uniform placement + the same GP reconstruction.
3. Adaptive placement + the same GP reconstruction.

Record organizer RMSE and resource usage across repeated episodes if competition rules allow. Without hidden ground truth, uncertainty alone cannot establish that adaptive placement improved accuracy.
